In [1]:

# Instalar Gurobi (si Colab no lo tiene)
!pip -q install gurobipy





[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#importaciones
import gurobipy as gp
from gurobipy import GRB
import time
import math

In [3]:
###############################################################
# CONFIGURACIÓN DEL MODELO
###############################################################

# Parámetros principales
rounds = 1         # Número de rondas
lanes = 5           # Keccak usa 5x5 lanes
bits = 4            # z = 4

# Tiempo máximo del solver
TIME_LIMIT = 1500

print("="*80)
print("MODELO MILP PARA ENCONTRAR EL MÍNIMO DE S-BOXES ACTIVAS EN KECCAK REDUCIDO")
print("="*80)

print("Configuración:")
print(f"  Rondas:          {rounds}")
print(f"  Lanes:           {lanes}x{lanes}")
print(f"  Bits por lane:   {bits}")
print(f"  Estado total:    {lanes*lanes*bits} bits")
print(f"  S-boxes/ronda:   {lanes*bits}")
print(f"  Tiempo límite:   {TIME_LIMIT} segundos ({TIME_LIMIT/60:.1f} minutos)")
print("="*80)

MODELO MILP PARA ENCONTRAR EL MÍNIMO DE S-BOXES ACTIVAS EN KECCAK REDUCIDO
Configuración:
  Rondas:          1
  Lanes:           5x5
  Bits por lane:   4
  Estado total:    100 bits
  S-boxes/ronda:   20
  Tiempo límite:   1500 segundos (25.0 minutos)


In [4]:
###############################################################
# TABLA DE ROTACIONES DE KECCAK
###############################################################

rotation_offsets = [

    [0, 36, 3, 41, 18],

    [1, 44, 10, 45, 2],

    [62, 6, 43, 15, 61],

    [28, 55, 25, 21, 56],

    [27, 20, 39, 8, 14]

]

In [5]:
###############################################################
# CREAR EL MODELO MILP
###############################################################

model = gp.Model("Keccak")

# Minimizar
model.ModelSense = GRB.MINIMIZE

# Tiempo límite
model.Params.TimeLimit = TIME_LIMIT

print("Modelo creado correctamente.")

Restricted license - for non-production use only - expires 2027-11-29
Set parameter TimeLimit to value 1500
Modelo creado correctamente.


In [6]:
###############################################################
# CREAR VARIABLES DEL MODELO
###############################################################

print("\nCreando variables del modelo...")

# ============================================================
# Estado principal
# S[r,x,y,z]
# ============================================================

S = model.addVars(

    rounds + 1,
    lanes,
    lanes,
    bits,

    vtype=GRB.BINARY,

    name="S"

)

# ============================================================
# Variables Theta
# C[r,x,z]
# ============================================================

C = model.addVars(

    rounds,
    lanes,
    bits,

    vtype=GRB.BINARY,

    name="C"

)

# ============================================================
# Variables Theta
# D[r,x,z]
# ============================================================

D = model.addVars(

    rounds,
    lanes,
    bits,

    vtype=GRB.BINARY,

    name="D"

)

# ============================================================
# Estado después de Theta
# B[r,x,y,z]
# ============================================================

B = model.addVars(

    rounds,
    lanes,
    lanes,
    bits,

    vtype=GRB.BINARY,

    name="B"

)

# ============================================================
# Entrada de Chi
# ============================================================

Chi_input = model.addVars(

    rounds,
    lanes,
    lanes,
    bits,

    vtype=GRB.BINARY,

    name="ChiInput"

)

# ============================================================
# S-box activa
# Una variable por cada operación Chi
# ============================================================

Chi_active = model.addVars(

    rounds,
    lanes,
    bits,

    vtype=GRB.BINARY,

    name="ChiActive"

)

# ============================================================
# Variables auxiliares AND
# ============================================================

AND = model.addVars(

    rounds,
    lanes,
    bits,
    lanes,

    vtype=GRB.BINARY,

    name="AND"

)

# ============================================================
# Contador de variables temporales XOR
# ============================================================

temp_counter = 0

print("Variables creadas correctamente.")

model.update()

print(f"Variables creadas: {model.NumVars}")


Creando variables del modelo...
Variables creadas correctamente.
Variables creadas: 560


In [7]:
###############################################################
# FUNCIONES AUXILIARES
###############################################################

print("\nCreando funciones auxiliares...")

# Contador global de variables temporales
temp_counter = 0


###############################################################
# Crear una nueva variable binaria temporal
###############################################################

def new_temp_var():

    global temp_counter

    nombre = f"TEMP_{temp_counter}"
    temp_counter += 1

    return model.addVar(
        vtype=GRB.BINARY,
        name=nombre
    )


###############################################################
# XOR ENTRE DOS VARIABLES
#
# z = x XOR y
###############################################################

def add_xor2(x, y):

    z = new_temp_var()

    model.addConstr(z <= x + y)

    model.addConstr(z >= x - y)

    model.addConstr(z >= y - x)

    model.addConstr(z <= 2 - x - y)

    return z


###############################################################
# XOR DE UNA LISTA DE VARIABLES
#
# Se construye de forma secuencial:
#
# (((a XOR b) XOR c) XOR d) ...
###############################################################

def add_xor_n(lista):

    if len(lista) == 1:
        return lista[0]

    resultado = lista[0]

    for i in range(1, len(lista)):

        resultado = add_xor2(resultado, lista[i])

    return resultado


###############################################################
# OR ENTRE UNA LISTA DE VARIABLES
#
# salida = OR(lista)
###############################################################

def add_or(lista):

    salida = new_temp_var()

    for v in lista:

        model.addConstr(salida >= v)

    model.addConstr(

        salida <= gp.quicksum(lista)

    )

    return salida


print("Funciones auxiliares creadas.")


Creando funciones auxiliares...
Funciones auxiliares creadas.


In [8]:
###############################################################
# RESTRICCIONES INICIALES
###############################################################

print("\nAgregando restricciones iniciales...")

#==============================================================
# Al menos un bit activo en el estado inicial
#==============================================================

model.addConstr(

    gp.quicksum(

        S[0, x, y, z]

        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)

    ) >= 1,

    name="EstadoInicial"

)

#==============================================================
# Al menos un bit activo en el estado final
#==============================================================

model.addConstr(

    gp.quicksum(

        S[rounds, x, y, z]

        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)

    ) >= 1,

    name="EstadoFinal"

)

print("✓ Restricciones iniciales agregadas.")


Agregando restricciones iniciales...
✓ Restricciones iniciales agregadas.


In [9]:
###############################################################
# RESTRICCIONES POR RONDA
###############################################################

print("\nAgregando restricciones por ronda...")

for r in range(rounds):

    print(f"  Procesando ronda {r+1}/{rounds}")

    ###########################################################
    # 1. PARIDAD DE COLUMNAS (THETA)
    #
    # C[x,z] = XOR de todos los bits de la columna
    ###########################################################

    for x in range(lanes):

        for z in range(bits):

            entradas = [

                S[r, x, y, z]

                for y in range(lanes)

            ]

            resultado = add_xor_n(entradas)

            model.addConstr(

                C[r, x, z] == resultado,

                name=f"Theta_C_{r}_{x}_{z}"

            )

    ###########################################################
    # 2. DIFUSIÓN (THETA)
    #
    # D[x,z] = C[x-1,z] XOR C[x+1,z-1]
    ###########################################################

    for x in range(lanes):

        for z in range(bits):

            a = C[r, (x-1) % lanes, z]

            b = C[r, (x+1) % lanes, (z-1) % bits]

            resultado = add_xor2(a, b)

            model.addConstr(

                D[r, x, z] == resultado,

                name=f"Theta_D_{r}_{x}_{z}"

            )

    ###########################################################
    # 3. THETA COMPLETA
    #
    # B = S XOR D
    ###########################################################

    for x in range(lanes):

        for y in range(lanes):

            for z in range(bits):

                resultado = add_xor2(

                    S[r, x, y, z],

                    D[r, x, z]

                )

                model.addConstr(

                    B[r, x, y, z] == resultado,

                    name=f"Theta_B_{r}_{x}_{y}_{z}"

                )

    ###########################################################
    # 4. RHO (ROTACIÓN) Y PI (PERMUTACIÓN)
    #
    # Aplicar la rotación de bits (Rho) y la permutación de
    # coordenadas (Pi) para obtener la entrada de Chi.
    ###########################################################

    for x in range(lanes):

        for y in range(lanes):

            #--------------------------------------------------
            # PI
            #
            # (x,y) -> (x',y')
            # x' = x + 3y (mod lanes)
            # y' = y
            #--------------------------------------------------

            x_prime = (x + 3 * y) % lanes
            y_prime = y

            #--------------------------------------------------
            # RHO
            #
            # Rotación de bits dentro del lane
            #--------------------------------------------------

            shift = rotation_offsets[x][y] % bits

            for z in range(bits):

                z_prime = (z - shift) % bits

                model.addConstr(

                    Chi_input[r, x_prime, y_prime, z]
                    ==
                    B[r, x, y, z_prime],

                    name=f"RhoPi_{r}_{x}_{y}_{z}"

                )

    
    ###########################################################
    # 5. CHI (OPERACIÓN NO LINEAL)
    #
    # Salida = Entrada XOR ((NOT Entrada+1) AND Entrada+2)
    #
    # Se linealiza utilizando variables auxiliares AND.
    ###########################################################

    for y in range(lanes):

        for z in range(bits):

            ###################################################
            # Cada fila corresponde a una S-box de 5 bits
            ###################################################

            for x in range(lanes):

                x1 = (x + 1) % lanes
                x2 = (x + 2) % lanes

                ################################################
                # Variable AND
                ################################################

                model.addConstr(

                    AND[r, y, z, x]
                    <=
                    Chi_input[r, x1, y, z]

                )

                model.addConstr(

                    AND[r, y, z, x]
                    <=
                    Chi_input[r, x2, y, z]

                )

                model.addConstr(

                    AND[r, y, z, x]
                    >=
                    Chi_input[r, x1, y, z]
                    +
                    Chi_input[r, x2, y, z]
                    - 1

                )

                ################################################
                # Salida = Entrada XOR AND
                ################################################

                salida = add_xor2(

                    Chi_input[r, x, y, z],

                    AND[r, y, z, x]

                )

                model.addConstr(

                    S[r+1, x, y, z] == salida,

                    name=f"Chi_{r}_{x}_{y}_{z}"

                )

            ####################################################
            # Chi activa
            #
            # Una S-box está activa si alguno de sus 5 bits
            # de entrada está activo.
            ####################################################

            input_bits = [

                Chi_input[r, x, y, z]

                for x in range(lanes)

            ]

            activa = add_or(input_bits)

            model.addConstr(

                Chi_active[r, y, z] == activa,

                name=f"ChiActive_{r}_{y}_{z}"

            )

print("  Restricciones completadas")
print("")


Agregando restricciones por ronda...
  Procesando ronda 1/1
  Restricciones completadas



In [10]:
# =============================================================================
# FUNCION OBJETIVO
# =============================================================================

# Minimizar el numero total de S-boxes activas en todas las rondas
objetivo = gp.quicksum(
    Chi_active[(r, y, z)]
    for r in range(rounds)
    for y in range(lanes)
    for z in range(bits)
)

model.setObjective(objetivo, GRB.MINIMIZE)

print("Funcion objetivo:")
print("  Minimizar: {0} S-boxes activas en {1} rondas".format(
    len([Chi_active[(r, y, z)]
         for r in range(rounds)
         for y in range(lanes)
         for z in range(bits)]),
    rounds))
print("")


# =============================================================================
# RESOLVER EL PROBLEMA
# =============================================================================

print("Resolviendo con Gurobi...")
print("Tiempo limite: {0} segundos".format(TIME_LIMIT))
print("")

start_time = time.time()

# Limite de tiempo
model.setParam('TimeLimit', TIME_LIMIT)

try:

    model.optimize()

    elapsed = time.time() - start_time


    # =========================================================================
    # ANALISIS DE RESULTADOS
    # =========================================================================

    print("")
    print("="*70)
    print("RESULTADOS DEL MILP")
    print("="*70)


    # Estado de solucion
    if model.Status == GRB.OPTIMAL:
        status = "OPTIMAL"

    elif model.Status == GRB.TIME_LIMIT:
        status = "TIME LIMIT"

    elif model.Status == GRB.INFEASIBLE:
        status = "INFEASIBLE"

    else:
        status = str(model.Status)


    print("Estado: {0}".format(status))
    print("Tiempo: {0:.2f} segundos".format(elapsed))


    # Valor objetivo
    if model.SolCount > 0:
        sboxes_activas = model.ObjVal
    else:
        sboxes_activas = None


    print("")
    print("COTA MINIMA DE S-BOXES ACTIVAS: {0}".format(sboxes_activas))


    # Detalle por ronda
    print("")
    print("Detalle por ronda:")

    for r in range(rounds):

        count = 0

        for y in range(lanes):
            for z in range(bits):

                val = Chi_active[(r,y,z)].X

                if val > 0.5:
                    count += 1


        print("  Ronda {0}: {1} S-boxes activas".format(r,count))


    # =========================================================================
    # ANALISIS DE PROBABILIDADES
    # =========================================================================

    print("")
    print("="*70)
    print("ANALISIS DE PROBABILIDADES")
    print("="*70)


    # Probabilidad maxima diferencial de Chi
    p_sbox = 0.5


    if sboxes_activas is not None:

        n = int(round(sboxes_activas))

        prob_total = p_sbox ** n

        pares_necesarios = 1.0 / prob_total


        print("Parametros de la S-box (chi de 5 bits):")
        print("  Probabilidad maxima por S-box: {0} (2/4)".format(p_sbox))
        print("")


        print("Trayectoria diferencial encontrada:")
        print("  S-boxes activas totales: {0}".format(n))
        print("  Probabilidad total:       (0.5)^{0} = {1:.2e}".format(
            n,prob_total))

        print("  Pares necesarios:         1/{0:.2e} = {1:.2e}".format(
            prob_total,pares_necesarios))

        print("")


        print("Interpretacion:")

        if n <= 10:
            print("  [ALTA PROBABILIDAD] Ataque factible")

        elif n <=20:
            print("  [PROBABILIDAD MEDIA] Requiere muchos pares")

        elif n <=30:
            print("  [BAJA PROBABILIDAD] Ataque dificil")

        else:
            print("  [PROBABILIDAD MUY BAJA]")
            print("  Inviable para Keccak completo")


        print("")


        print("Comparacion con Keccak-f[1600] (24 rondas):")
        print("  Keccak completo tiene 24 rondas y 320 S-boxes por ronda")
        print("  El numero de S-boxes activas aumenta con las rondas")
        print("  Ataques diferenciales completos son inviables")


    # =========================================================================
    # VARIABLES ACTIVAS RONDA 0
    # =========================================================================


    print("")
    print("="*70)
    print("VARIABLES ACTIVAS EN RONDA 0 (primeros 3 bits)")
    print("="*70)


    count_active = 0


    for x in range(lanes):
        for y in range(lanes):
            for z in range(min(3,bits)):


                val = S[(0,x,y,z)].X


                if val > 0.5:

                    print(
                    "  S[0,{0},{1},{2}] = 1".format(x,y,z)
                    )

                    count_active += 1


    if count_active == 0:
        print("  (No se encontraron variables activas)")


    # =========================================================================
    # GUARDAR RESULTADOS
    # =========================================================================


    filename = "resultados_keccak_r{0}_z{1}.txt".format(
        rounds,bits)


    with open(filename,"w") as f:


        f.write("="*70+"\n")
        f.write("RESULTADOS DEL MODELO MILP PARA KECCAK\n")
        f.write("="*70+"\n\n")


        f.write("CONFIGURACION:\n")
        f.write("  Rondas: {}\n".format(rounds))
        f.write("  Lanes: {}x{}\n".format(lanes,lanes))
        f.write("  Bits por lane: {}\n".format(bits))
        f.write("  Estado total: {} bits\n".format(
            lanes*lanes*bits))

        f.write("  Tiempo limite: {}\n\n".format(
            TIME_LIMIT))


        f.write("RESULTADOS:\n")
        f.write("  Estado: {}\n".format(status))
        f.write("  Tiempo: {:.2f}\n".format(elapsed))
        f.write("  S-boxes activas: {}\n\n".format(
            sboxes_activas))


        f.write("DETALLE POR RONDA:\n")

        for r in range(rounds):

            count=0

            for y in range(lanes):
                for z in range(bits):

                    if Chi_active[(r,y,z)].X >0.5:
                        count+=1


            f.write(
            "  Ronda {}: {} S-boxes activas\n".format(
                r,count))


    print("")
    print("Resultados guardados en: {}".format(filename))
    print("="*70)


except Exception as e:

    print("")
    print("ERROR al resolver:")
    print(e)

    print("")
    print("Sugerencias:")
    print("  1. Reduce rounds a 2")
    print("  2. Reduce bits a 2")
    print("  3. Aumenta TIME_LIMIT")
    print("  4. Usa licencia Gurobi completa")


print("")
print("Fin del modelo.")

Funcion objetivo:
  Minimizar: 20 S-boxes activas en 1 rondas

Resolviendo con Gurobi...
Tiempo limite: 1500 segundos

Set parameter TimeLimit to value 1500
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1500

Optimize a model with 1982 rows, 880 columns and 5540 nonzeros (Min)
Model fingerprint: 0x9043d0a9
Model has 20 linear objective coefficients
Variable types: 0 continuous, 880 integer (880 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Presolve removed 780 rows and 500 columns
Presolve time: 0.04s
Presolved: 1202 rows, 380 columns, 4120 nonzeros
Variable types: 0 continuous, 380 integer (380 binary)
Found heuristic solution: obje